**AI Statement**: This entire repository and project was created with assistance from AI tools (Claude/Cursor) for code generation, documentation, and analysis. All analysis results, interpretations, and conclusions are the work of the author.


# Warhammer 40K Character Network - Explainer Notebook


## Table of Contents

1. [Code Overview](#code-overview)
   - [Network Construction](#1-createnetworkpy---network-construction)
   - [Network Topology Analysis](#2-analysespy---network-topology-analysis)
   - [Semantic Clustering](#3-semanticspy---semantic-clustering)
   - [Mythology vs Narrative Analysis](#4-mythologyanalysispy---mythology-vs-narrative-analysis)
   - [Portal vs Cluster Comparison](#5-compareportalsclusterspy---portal-vs-cluster-comparison)
2. [Main Results](#main-results)
3. [Data Cleaning and Preprocessing](#data-cleaning-and-preprocessing)
   - [Character Filtering](#character-filtering)
   - [Network Filtering](#network-filtering)
   - [Text Cleaning for Semantic Analysis](#text-cleaning-for-semantic-analysis)
   - [Edge Extraction](#edge-extraction)
4. [Additional Analyses](#additional-analyses)
   - [Betweenness Centrality Distribution](#betweenness-centrality-distribution)
   - [Semantic Cluster Size Distribution](#semantic-cluster-size-distribution)
   - [Portal Assignment Statistics](#portal-assignment-statistics)
   - [Network Density and Clustering](#network-density-and-clustering)
   - [Character Name Variations](#character-name-variations)


In [101]:
import pickle
import json
import pandas as pd
import networkx as nx
from pathlib import Path

# Set up paths
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data"
IMAGES_PATH = BASE_DIR / "images"


## Code Overview {#code-overview}

### 1. `create_network.py` - Network Construction {#1-createnetworkpy---network-construction}
Builds the character network from raw wiki data:
- Maps characters to their faction affiliations
- Extracts character entries from wiki pages
- Creates edges between characters based on wiki links
- Builds a NetworkX graph with node and edge attributes
- Filters to largest connected component (removes isolated nodes)


### 2. `Analyses.py` - Network Topology Analysis {#2-analysespy---network-topology-analysis}
Analyzes the network structure:
- **Degree distribution**: How many connections each character has
- **Centrality measures**: Degree, betweenness, and eigenvector centrality
- **Assortativity**: Whether high-degree characters connect to other high-degree characters
- **Community detection**: Uses Louvain algorithm to find communities and compares with factions
- **Network backbone**: Extracts the most important edges


### 3. `semantics.py` - Semantic Clustering {#3-semanticspy---semantic-clustering}
Groups characters by text similarity:
- Loads character descriptions from wiki pages
- Generates text embeddings using sentence transformers
- Finds optimal number of clusters using silhouette score
- Assigns each character to a semantic cluster based on their description text


### 4. `mythology_analysis.py` - Mythology vs Narrative Analysis {#4-mythologyanalysispy---mythology-vs-narrative-analysis}
Combines topology and semantics to identify:
- **Topology metrics**: Computes betweenness centrality, PageRank, clustering coefficient, and network communities
- **Neighbor semantic consistency**: Calculates how similar each character's text is to their neighbors' texts
- **Mythological anchors**: High betweenness (bridge many connections) but low semantic consistency (don't share themes with neighbors)
- **Narrative agents**: Low betweenness but high semantic consistency (share themes with neighbors)
- Analyzes how network position relates to semantic content


### 5. `compare_portals_clusters.py` - Portal vs Cluster Comparison {#5-compareportalsclusterspy---portal-vs-cluster-comparison}
Compares faction-based groupings (portals) with semantic clusters:
- Creates confusion matrices showing how factions map to semantic clusters
- Calculates alignment metrics between portals and clusters


## Main Results {#main-results}


In [102]:
# Load the filtered network
with open(DATA_PATH / "lexicanum_network_filtered.pkl", 'rb') as f:
    G = pickle.load(f)

# Convert to undirected for analysis
if isinstance(G, nx.DiGraph):
    G_undirected = G.to_undirected()
else:
    G_undirected = G

print("NETWORK STATISTICS")
print(f"Nodes (characters): {G_undirected.number_of_nodes():,}")
print(f"Edges (connections): {G_undirected.number_of_edges():,}")
print(f"Density: {nx.density(G_undirected):.6f}")
print(f"Average degree: {sum(dict(G_undirected.degree()).values()) / G_undirected.number_of_nodes():.2f}")
print()


NETWORK STATISTICS
Nodes (characters): 3,668
Edges (connections): 9,409
Density: 0.001399
Average degree: 5.13



In [103]:
# Degree distribution
degrees = [G_undirected.degree(node) for node in G_undirected.nodes()]
degrees_dict = dict(G_undirected.degree())

print("DEGREE DISTRIBUTION")
print(f"Mean degree: {sum(degrees) / len(degrees):.2f}")
print(f"Median degree: {sorted(degrees)[len(degrees)//2]}")
print(f"Max degree: {max(degrees)}")
print(f"Min degree: {min(degrees)}")
print()

# Top characters by degree
top_degree = sorted(degrees_dict.items(), key=lambda x: x[1], reverse=True)[:50]
print("Top 50 characters by degree (most connections):")
for i, (char, deg) in enumerate(top_degree, 1):
    print(f"  {i:2d}. {char:40s} - {deg} connections")
print()


DEGREE DISTRIBUTION
Mean degree: 5.13
Median degree: 3
Max degree: 161
Min degree: 1

Top 50 characters by degree (most connections):
   1. Roboute Guilliman                        - 161 connections
   2. Emperor of Mankind                       - 150 connections
   3. Rogal Dorn                               - 127 connections
   4. Ibram Gaunt                              - 126 connections
   5. Angron                                   - 121 connections
   6. Sanguinius                               - 113 connections
   7. Lion El'Jonson                           - 108 connections
   8. Fulgrim                                  - 91 connections
   9. Ezekyle Abaddon                          - 86 connections
  10. Vulkan                                   - 84 connections
  11. Fabius Bile                              - 80 connections
  12. Mortarion                                - 77 connections
  13. Leman Russ (Primarch)                    - 76 connections
  14. Malcador the Sigillit

In [104]:
# Centrality measures
print("CENTRALITY ANALYSIS")

betweenness = nx.betweenness_centrality(G_undirected)
eigenvector = nx.eigenvector_centrality(G_undirected, max_iter=1000)
degree_centrality = nx.degree_centrality(G_undirected)

# Top characters by betweenness
top_betweenness = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop 10 characters by betweenness centrality (bridge characters):")
for i, (char, val) in enumerate(top_betweenness, 1):
    print(f"  {i:2d}. {char:40s} - {val:.6f}")
print()

# Top characters by eigenvector centrality
top_eigenvector = sorted(eigenvector.items(), key=lambda x: x[1], reverse=True)[:50]
print("Top 10 characters by eigenvector centrality (connected to important characters):")
for i, (char, val) in enumerate(top_eigenvector, 1):
    print(f"  {i:2d}. {char:40s} - {val:.6f}")
print()


CENTRALITY ANALYSIS

Top 10 characters by betweenness centrality (bridge characters):
   1. Emperor of Mankind                       - 0.301267
   2. Roboute Guilliman                        - 0.216343
   3. Rogal Dorn                               - 0.083584
   4. Saint Sabbat                             - 0.076297
   5. Angron                                   - 0.072689
   6. Lion El'Jonson                           - 0.065926
   7. Vulkan                                   - 0.062737
   8. Ezekyle Abaddon                          - 0.055541
   9. Leman Russ (Primarch)                    - 0.055397
  10. Ibram Gaunt                              - 0.053721

Top 10 characters by eigenvector centrality (connected to important characters):
   1. Emperor of Mankind                       - 0.248804
   2. Roboute Guilliman                        - 0.242769
   3. Rogal Dorn                               - 0.232547
   4. Sanguinius                               - 0.204107
   5. Horus Lupercal

In [105]:
# Assortativity
assortativity = nx.degree_assortativity_coefficient(G_undirected)
print("ASSORTATIVITY")
print(f"Degree assortativity coefficient: {assortativity:.4f}")

ASSORTATIVITY
Degree assortativity coefficient: -0.0195


In [106]:
# Community detection
print("COMMUNITY DETECTION")

communities = nx.community.louvain_communities(G_undirected, seed=42)
modularity = nx.community.modularity(G_undirected, communities)

print(f"Number of communities found: {len(communities)}")
print(f"Modularity score: {modularity:.4f}")
print()

# Largest communities
community_sizes = sorted([len(comm) for comm in communities], reverse=True)
print("All communities:")
for i, size in enumerate(community_sizes, 1):
    print(f"  {i:2d}. {size} characters")
print()


COMMUNITY DETECTION
Number of communities found: 42
Modularity score: 0.7879

All communities:
   1. 362 characters
   2. 294 characters
   3. 230 characters
   4. 210 characters
   5. 210 characters
   6. 207 characters
   7. 180 characters
   8. 176 characters
   9. 169 characters
  10. 143 characters
  11. 125 characters
  12. 120 characters
  13. 108 characters
  14. 98 characters
  15. 96 characters
  16. 95 characters
  17. 79 characters
  18. 76 characters
  19. 75 characters
  20. 70 characters
  21. 68 characters
  22. 54 characters
  23. 47 characters
  24. 45 characters
  25. 43 characters
  26. 35 characters
  27. 32 characters
  28. 32 characters
  29. 21 characters
  30. 21 characters
  31. 20 characters
  32. 17 characters
  33. 16 characters
  34. 14 characters
  35. 13 characters
  36. 12 characters
  37. 12 characters
  38. 11 characters
  39. 9 characters
  40. 9 characters
  41. 8 characters
  42. 6 characters



In [107]:
# Semantic clustering results
print("SEMANTIC CLUSTERING")

with open(DATA_PATH / "optimal_semantic_clusters.json", 'r', encoding='utf-8') as f:
    semantic_data = json.load(f)

print(f"Optimal number of clusters: {semantic_data['optimal_k']}")
print(f"Total characters clustered: {semantic_data['total_characters']}")
print(f"Silhouette score: {semantic_data['silhouette_score']:.4f}")
print()

# Load cluster summaries
with open(DATA_PATH / "optimal_cluster_summaries.json", 'r', encoding='utf-8') as f:
    cluster_summaries = json.load(f)

print("All semantic clusters (by size):")
clusters_sorted = sorted(
    cluster_summaries['clusters'].items(),
    key=lambda x: x[1]['size'],
    reverse=True
)

for i, (cluster_id, info) in enumerate(clusters_sorted, 1):
    print(f"  {i:2d}. Cluster {cluster_id}: {info['size']} characters")
    print(f"      Keywords: {info['keywords']}")
print()


SEMANTIC CLUSTERING
Optimal number of clusters: 30
Total characters clustered: 3667
Silhouette score: 0.0412

All semantic clusters (by size):
   1. Cluster 18: 242 characters
      Keywords: warband, legion, scythes, tearers, sorcerer
   2. Cluster 15: 217 characters
      Keywords: angels, legion, caliban, chaplain, sergeant
   3. Cluster 7: 196 characters
      Keywords: ordo, retinue, xenos, malleus, inquisition
   4. Cluster 24: 196 characters
      Keywords: legion, regiment, general, colonel, worlds
   5. Cluster 13: 192 characters
      Keywords: ultramarines, calth, bearers, macragge, sergeant
   6. Cluster 12: 187 characters
      Keywords: cadian, regiment, general, legion, cadia
   7. Cluster 8: 184 characters
      Keywords: legion, primarch, terra, m30, angels
   8. Cluster 9: 162 characters
      Keywords: legion, eaters, warband, khorne, dreadnought
   9. Cluster 6: 151 characters
      Keywords: wolves, fenris, pack, priest, prospero
  10. Cluster 22: 139 characters
  

In [119]:
# Mythology vs Narrative analysis
print("MYTHOLOGY VS NARRATIVE ANALYSIS")

if (DATA_PATH / "mythology_vs_narrative_analysis.csv").exists():
    df = pd.read_csv(DATA_PATH / "mythology_vs_narrative_analysis.csv")
    
    print(f"Characters analyzed: {len(df)}")
    print(f"Semantic clusters: {df['semantic_cluster_id'].nunique()}")
    print(f"Network communities: {df['network_community_id'].nunique()}")
    print()
    
    print("Betweenness Centrality:")
    print(f"  Mean: {df['betweenness'].mean():.6f}")
    print(f"  Median: {df['betweenness'].median():.6f}")
    print(f"  Max: {df['betweenness'].max():.6f}")
    print()
    
    print("Neighbor Semantic Consistency:")
    print(f"  Mean: {df['neighbor_consistency'].mean():.4f}")
    print(f"  Median: {df['neighbor_consistency'].median():.4f}")
    print(f"  Min: {df['neighbor_consistency'].min():.4f}")
    print(f"  Max: {df['neighbor_consistency'].max():.4f}")
    print()
    
    # Calculate correlation between betweenness and neighbor consistency
    clean_df = df.dropna(subset=['betweenness', 'neighbor_consistency'])
    if len(clean_df) > 0:
        correlation = clean_df['betweenness'].corr(clean_df['neighbor_consistency'], method='pearson')
        print("Correlation (Betweenness vs Consistency):")
        print(f"  r = {correlation:.4f}")
    
    # Identify mythological anchors (high betweenness, low consistency)
    betweenness_threshold = df['betweenness'].quantile(0.90)
    consistency_threshold = df['neighbor_consistency'].quantile(0.10)
    
    anchors = df[
        (df['betweenness'] >= betweenness_threshold) & 
        (df['neighbor_consistency'] <= consistency_threshold)
    ].sort_values('betweenness', ascending=False)
    
    print(f"Mythological anchors (high betweenness, low semantic consistency):")
    print(f"  Found {len(anchors)} candidates")
    print(f"  Thresholds: betweenness >= {betweenness_threshold:.6f}, consistency <= {consistency_threshold:.4f}")
    print()
    
    if len(anchors) > 0:
        print("Top 10 mythological anchor candidates:")
        for i, (idx, row) in enumerate(anchors.head(10).iterrows(), 1):
            print(f"  {i:2d}. {row['node_id']:40s}")
            print(f"      Betweenness: {row['betweenness']:.6f}, Consistency: {row['neighbor_consistency']:.4f}")
        print()
        
        if len(anchors) > 10:
            print("Bottom 10 mythological anchor candidates:")
            for i, (idx, row) in enumerate(anchors.tail(10).iterrows(), 1):
                print(f"  {i:2d}. {row['node_id']:40s}")
                print(f"      Betweenness: {row['betweenness']:.6f}, Consistency: {row['neighbor_consistency']:.4f}")
    print()
print()


MYTHOLOGY VS NARRATIVE ANALYSIS
Characters analyzed: 3667
Semantic clusters: 30
Network communities: 50

Betweenness Centrality:
  Mean: 0.000475
  Median: 0.000000
  Max: 0.085229

Neighbor Semantic Consistency:
  Mean: 0.5800
  Median: 0.5733
  Min: 0.0924
  Max: 1.0000

Correlation (Betweenness vs Consistency):
  r = -0.0522
Mythological anchors (high betweenness, low semantic consistency):
  Found 24 candidates
  Thresholds: betweenness >= 0.000683, consistency <= 0.4265

Top 10 mythological anchor candidates:
   1. Saint Sabbat                            
      Betweenness: 0.012788, Consistency: 0.4053
   2. Kor'sarro Khan                          
      Betweenness: 0.010658, Consistency: 0.4075
   3. Hyperion                                
      Betweenness: 0.009848, Consistency: 0.3511
   4. Auric Stormcloud                        
      Betweenness: 0.003877, Consistency: 0.4180
   5. Janus Darke                             
      Betweenness: 0.003441, Consistency: 0.4182


In [109]:
# Faction distribution
print("FACTION DISTRIBUTION")

faction_counts = {}
for node in G_undirected.nodes():
    portal = G_undirected.nodes[node].get('portals', '')
    if portal:
        portal_str = str(portal).strip()
        if portal_str:
            faction_counts[portal_str] = faction_counts.get(portal_str, 0) + 1

if faction_counts:
    sorted_factions = sorted(faction_counts.items(), key=lambda x: x[1], reverse=True)
    print("Characters by faction:")
    for i, (faction, count) in enumerate(sorted_factions, 1):
        print(f"  {i:2d}. {faction:20s} - {count:4d} characters")
else:
    print("No faction data available.")
print()


FACTION DISTRIBUTION
Characters by faction:
   1. Space Marines        - 1334 characters
   2. Chaos                -  868 characters
   3. Imperium             -  531 characters
   4. Astra Militarum      -  348 characters
   5. Inquisition          -  176 characters
   6. Mechanicum           -   97 characters
   7. Asuryani             -   73 characters
   8. Adeptus Custodes     -   60 characters
   9. Adepta Sororitas     -   55 characters
  10. Necrons              -   49 characters
  11. Drukhari             -   45 characters
  12. T'au                 -   29 characters
  13. Leagues of Votann    -    2 characters



## Data Cleaning and Preprocessing {#data-cleaning-and-preprocessing}

This section details the choices made during data cleaning and preprocessing steps.

### Character Filtering {#character-filtering}

We filter out non-character pages using pattern matching. Pages matching patterns like "List of...", "Known members of...", or "Category:" are excluded as they represent organizational pages rather than individual characters.


In [110]:
# Character filtering patterns
import re

CHARACTER_EXCLUDE_PATTERNS = [
    r'^list of', r'^known members of', r'^known captains of',
    r'^members of', r'^characters of', r'formation', r'^category:',
    r'\(list\)$', r'\(list\)'
]

def is_character(title):
    title_lower = title.lower()
    for pattern in CHARACTER_EXCLUDE_PATTERNS:
        if re.search(pattern, title_lower):
            return False
    return True

# Example: Check some titles
test_titles = [
    "Roboute Guilliman",
    "List of Space Marine Chapters",
    "Known Members of the Inquisition",
    "Angron"
]

print("Character filtering examples:")
for title in test_titles:
    result = is_character(title)
    print(f"  '{title}': {'Character' if result else 'Excluded'}")


Character filtering examples:
  'Roboute Guilliman': Character
  'List of Space Marine Chapters': Excluded
  'Known Members of the Inquisition': Excluded
  'Angron': Character


### Network Filtering {#network-filtering}

The raw network contains many isolated nodes and small disconnected components. We apply two filtering steps:

1. **Largest Connected Component**: Extract only the largest weakly connected component to ensure all characters are reachable from each other
2. **Degree Filtering**: Remove nodes with degree < 2 iteratively until stable. This removes "dead ends" that don't contribute to the network structure


In [111]:
# Network filtering statistics
with open(DATA_PATH / "lexicanum_network.pkl", 'rb') as f:
    G_full = pickle.load(f)

print("NETWORK FILTERING STATISTICS")
print(f"Original network:")
print(f"  Nodes: {G_full.number_of_nodes():,}")
print(f"  Edges: {G_full.number_of_edges():,}")

# Count components
components = list(nx.weakly_connected_components(G_full))
print(f"  Connected components: {len(components)}")
component_sizes = [len(c) for c in components]
component_sizes.sort(reverse=True)
print(f"  Largest component: {component_sizes[0]:,} nodes")
print(f"  Second largest: {component_sizes[1] if len(component_sizes) > 1 else 0:,} nodes")

# Load filtered network
with open(DATA_PATH / "lexicanum_network_filtered.pkl", 'rb') as f:
    G_filtered = pickle.load(f)

print(f"\nFiltered network:")
print(f"  Nodes: {G_filtered.number_of_nodes():,}")
print(f"  Edges: {G_filtered.number_of_edges():,}")
print(f"  Reduction: {((1 - G_filtered.number_of_nodes() / G_full.number_of_nodes()) * 100):.1f}% of nodes removed")
print(f"  Reduction: {((1 - G_filtered.number_of_edges() / G_full.number_of_edges()) * 100):.1f}% of edges removed")


NETWORK FILTERING STATISTICS
Original network:
  Nodes: 12,399
  Edges: 15,858
  Connected components: 5872
  Largest component: 5,207 nodes
  Second largest: 21 nodes

Filtered network:
  Nodes: 3,668
  Edges: 11,997
  Reduction: 70.4% of nodes removed
  Reduction: 24.3% of edges removed


### Text Cleaning for Semantic Analysis {#text-cleaning-for-semantic-analysis}

For semantic clustering, we clean wiki text by:
1. Removing reference sections (Sources, References, etc.)
2. Removing wiki markup (templates, links, headers)
3. Removing character names to focus on descriptive content rather than mentions
4. Normalizing whitespace and converting to lowercase


In [112]:
# Example of text cleaning
example_wikitext = """
== Biography ==
[[Roboute Guilliman]] was a [[Primarch]] of the [[Ultramarines]] Legion.
He fought in the [[Horus Heresy]] alongside [[Lion El'Jonson]].

{{SpaceMarinePortal}}

== Sources ==
* Codex: Space Marines
"""

print("Original wikitext:")
print(example_wikitext)
print("\nAfter cleaning (removing markup, references, character names):")
print("(This would be processed by clean_wikitext function)")
print("Result: 'biography was a primarch of the legion fought in the alongside'")
print("\nNote: Character names are removed to focus on descriptive content")


Original wikitext:

== Biography ==
[[Roboute Guilliman]] was a [[Primarch]] of the [[Ultramarines]] Legion.
He fought in the [[Horus Heresy]] alongside [[Lion El'Jonson]].

{{SpaceMarinePortal}}

== Sources ==
* Codex: Space Marines


After cleaning (removing markup, references, character names):
(This would be processed by clean_wikitext function)
Result: 'biography was a primarch of the legion fought in the alongside'

Note: Character names are removed to focus on descriptive content


### Edge Extraction {#edge-extraction}

Edges are created when one character's wiki page links to another character's page. We:
1. Extract all wiki links from each character's page
2. Filter out links to portals, categories, or special pages (containing ":")
3. Match links to character titles using name variations (spaces vs underscores, etc.)
4. Count multiple links between the same characters as edge weight


In [113]:
# Edge statistics
with open(DATA_PATH / "lexicanum_edges.json", 'r', encoding='utf-8') as f:
    edges_data = json.load(f)

# Count edge weights (multiple links between same characters)
edge_weights = {}
for edge in edges_data:
    source = edge['source']
    target = edge['target']
    key = (source, target)
    if key in edge_weights:
        edge_weights[key] += 1
    else:
        edge_weights[key] = 1

weight_distribution = {}
for weight in edge_weights.values():
    if weight in weight_distribution:
        weight_distribution[weight] += 1
    else:
        weight_distribution[weight] = 1

print("EDGE WEIGHT DISTRIBUTION")
print("(Number of times same character pair is linked)")
for weight in sorted(weight_distribution.keys()):
    count = weight_distribution[weight]
    print(f"  Weight {weight}: {count:,} edges")
    
print(f"\nTotal unique character pairs: {len(edge_weights):,}")
print(f"Total links (with duplicates): {len(edges_data):,}")
print(f"Average links per pair: {len(edges_data) / len(edge_weights):.2f}")


EDGE WEIGHT DISTRIBUTION
(Number of times same character pair is linked)
  Weight 1: 13,939 edges
  Weight 2: 1,487 edges
  Weight 3: 315 edges
  Weight 4: 76 edges
  Weight 5: 26 edges
  Weight 6: 11 edges
  Weight 7: 2 edges
  Weight 8: 1 edges
  Weight 12: 1 edges

Total unique character pairs: 15,858
Total links (with duplicates): 18,392
Average links per pair: 1.16


## Additional Analyses {#additional-analyses}

This section contains analyses and statistics that didn't make it into the main text but provide additional insights.


### Betweenness Centrality Distribution {#betweenness-centrality-distribution}

Betweenness centrality shows a highly skewed distribution - most characters have very low betweenness, while a few key characters act as bridges connecting different parts of the network.


In [114]:
# Betweenness distribution analysis
if (DATA_PATH / "mythology_vs_narrative_analysis.csv").exists():
    df = pd.read_csv(DATA_PATH / "mythology_vs_narrative_analysis.csv")
    
    betweenness = df['betweenness'].tolist()
    betweenness.sort()
    
    print("BETWEENNESS CENTRALITY DISTRIBUTION")
    print(f"Total characters: {len(betweenness)}")
    print(f"Mean: {sum(betweenness) / len(betweenness):.6f}")
    print(f"Median: {betweenness[len(betweenness)//2]:.6f}")
    print(f"Max: {betweenness[-1]:.6f}")
    print(f"Min: {betweenness[0]:.6f}")
    print()
    
    # Percentiles
    percentiles = [10, 25, 50, 75, 90, 95, 99]
    print("Percentiles:")
    for p in percentiles:
        idx = int(len(betweenness) * p / 100)
        if idx >= len(betweenness):
            idx = len(betweenness) - 1
        print(f"  {p}th percentile: {betweenness[idx]:.6f}")
    
    # Count characters with zero betweenness
    zero_count = sum(1 for b in betweenness if b == 0)
    print(f"\nCharacters with zero betweenness: {zero_count} ({zero_count/len(betweenness)*100:.1f}%)")
    print("(These characters don't act as bridges between other characters)")
else:
    print("Mythology vs narrative analysis file not found.")


BETWEENNESS CENTRALITY DISTRIBUTION
Total characters: 3667
Mean: 0.000475
Median: 0.000000
Max: 0.085229
Min: 0.000000

Percentiles:
  10th percentile: 0.000000
  25th percentile: 0.000000
  50th percentile: 0.000000
  75th percentile: 0.000003
  90th percentile: 0.000683
  95th percentile: 0.002128
  99th percentile: 0.010363

Characters with zero betweenness: 2656 (72.4%)
(These characters don't act as bridges between other characters)


### Semantic Cluster Size Distribution {#semantic-cluster-size-distribution}

The semantic clusters vary significantly in size. Some clusters contain many characters with similar themes, while others are smaller and more specialized.


In [115]:
# Semantic cluster size distribution
with open(DATA_PATH / "optimal_cluster_summaries.json", 'r', encoding='utf-8') as f:
    cluster_summaries = json.load(f)

cluster_sizes = []
for cluster_id in cluster_summaries['clusters'].keys():
    size = cluster_summaries['clusters'][cluster_id]['size']
    cluster_sizes.append(size)

cluster_sizes.sort(reverse=True)

print("SEMANTIC CLUSTER SIZE DISTRIBUTION")
print(f"Total clusters: {len(cluster_sizes)}")
print(f"Total characters: {sum(cluster_sizes)}")
print(f"Mean cluster size: {sum(cluster_sizes) / len(cluster_sizes):.1f}")
print(f"Largest cluster: {cluster_sizes[0]} characters")
print(f"Smallest cluster: {cluster_sizes[-1]} characters")
print()
print("Cluster size distribution:")
size_ranges = [(1, 10), (11, 25), (26, 50), (51, 100), (101, 200), (201, 300)]
for min_size, max_size in size_ranges:
    count = sum(1 for s in cluster_sizes if min_size <= s <= max_size)
    if count > 0:
        print(f"  {min_size}-{max_size} characters: {count} clusters")


SEMANTIC CLUSTER SIZE DISTRIBUTION
Total clusters: 30
Total characters: 3667
Mean cluster size: 122.2
Largest cluster: 242 characters
Smallest cluster: 26 characters

Cluster size distribution:
  26-50 characters: 3 clusters
  51-100 characters: 9 clusters
  101-200 characters: 16 clusters
  201-300 characters: 2 clusters


### Portal Assignment Statistics {#portal-assignment-statistics}

Not all characters have portal (faction) assignments. Some characters may have multiple portals, or none at all if they don't clearly belong to a main faction.


In [116]:
# Portal assignment statistics
characters_with_portal = 0
characters_without_portal = 0
multi_portal_count = 0

for node in G_undirected.nodes():
    portal = G_undirected.nodes[node].get('portals', '')
    if portal:
        portal_str = str(portal).strip()
        if portal_str:
            characters_with_portal += 1
            # Check if multiple portals (comma-separated)
            if ',' in portal_str:
                multi_portal_count += 1
        else:
            characters_without_portal += 1
    else:
        characters_without_portal += 1

total = G_undirected.number_of_nodes()
print("PORTAL ASSIGNMENT STATISTICS")
print(f"Total characters: {total:,}")
print(f"Characters with portal: {characters_with_portal:,} ({characters_with_portal/total*100:.2f}%)")


PORTAL ASSIGNMENT STATISTICS
Total characters: 3,668
Characters with portal: 3,667 (99.97%)


### Network Density and Clustering {#network-density-and-clustering}

The network is quite sparse (low density), which is typical for social networks. Most characters are only connected to a small subset of all possible characters.


In [117]:
# Network density and clustering statistics
print("NETWORK STRUCTURE STATISTICS")
print(f"Density: {nx.density(G_undirected):.6f}")
print(f"(Density of 1.0 would mean every character links to every other character)")
print()

# Average clustering coefficient
clustering = nx.clustering(G_undirected)
clustering_values = list(clustering.values())
avg_clustering = sum(clustering_values) / len(clustering_values) if clustering_values else 0
print(f"Average clustering coefficient: {avg_clustering:.4f}")
print(f"(Measures how connected a character's neighbors are to each other)")
print(f"  Range: 0 (no triangles) to 1 (fully connected neighbors)")
print()

# Path length statistics
print("Average shortest path length (sampling 1000 random pairs):")
sample_size = min(1000, G_undirected.number_of_nodes())
sample_nodes = list(G_undirected.nodes())[:sample_size]
path_lengths = []
for i in range(min(100, len(sample_nodes))):
    for j in range(i + 1, min(i + 10, len(sample_nodes))):
        try:
            path_len = nx.shortest_path_length(G_undirected, sample_nodes[i], sample_nodes[j])
            path_lengths.append(path_len)
        except nx.NetworkXNoPath:
            pass

if path_lengths:
    print(f"  Mean: {sum(path_lengths) / len(path_lengths):.2f} steps")
    print(f"  Max: {max(path_lengths)} steps")
    print(f"  (Based on {len(path_lengths)} sampled paths)")


NETWORK STRUCTURE STATISTICS
Density: 0.001399
(Density of 1.0 would mean every character links to every other character)

Average clustering coefficient: 0.5075
(Measures how connected a character's neighbors are to each other)
  Range: 0 (no triangles) to 1 (fully connected neighbors)

Average shortest path length (sampling 1000 random pairs):
  Mean: 5.24 steps
  Max: 14 steps
  (Based on 900 sampled paths)


### Character Name Variations {#character-name-variations}

Wiki pages use different name formats (spaces, underscores, CamelCase). We handle these variations when matching links to characters to ensure we capture all connections.


In [118]:
# Example of name variation handling
example_names = [
    "Roboute Guilliman",
    "Roboute_Guilliman",
    "RobouteGuilliman",
    "SpaceMarinePortal",
    "Space Marine Portal"
]

print("NAME VARIATION HANDLING")
print("Wiki pages use different formats for the same entity:")
for name in example_names:
    # Simulate the normalization process
    if "Portal" in name:
        cleaned = name.replace("Portal", "").strip()
        print(f"  '{name}' -> '{cleaned}' (after removing 'Portal' suffix)")
    else:
        print(f"  '{name}' (kept as-is or normalized)")
        
print("\nThese variations are all matched to the same character when extracting edges.")


NAME VARIATION HANDLING
Wiki pages use different formats for the same entity:
  'Roboute Guilliman' (kept as-is or normalized)
  'Roboute_Guilliman' (kept as-is or normalized)
  'RobouteGuilliman' (kept as-is or normalized)
  'SpaceMarinePortal' -> 'SpaceMarine' (after removing 'Portal' suffix)
  'Space Marine Portal' -> 'Space Marine' (after removing 'Portal' suffix)

These variations are all matched to the same character when extracting edges.


# End
